# 4 · LCEL Chains — Draft Reply & QA/Summary

Once the specialists have gathered facts, two more **deterministic LCEL chains** finish the job:

1. **Draft reply** — write the customer-facing response in brand voice.
2. **QA / summary** — produce a short internal summary for the CRM.

In [1]:
%run aurora_common.py

C:\Users\akhawaja\git\cs4603\wk4_capstone\aurora_common.py:42: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Draft-reply chain
> **TODO (student):** enforce brand voice (friendly, concise, no internal jargon) and always include
> next steps.

In [2]:
draft_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are Aurora, a friendly support agent for an online store. "
     "Write a concise, warm reply to the customer using the research provided. "
     "Do not invent facts beyond the research."),
    ("human",
     "Customer message:\n{ticket}\n\nResearch / facts:\n{research}\n\nWrite the reply:"),
])

draft_chain = draft_prompt | llm_noreason | StrOutputParser()

reply = draft_chain.invoke({
    "ticket": "Where is my order 1001?",
    "research": "Order 1001 is shipped; in transit; expected in 2 days via ACME Express.",
})
print(reply)

Hello! Thanks for reaching out.

Good news—your order #1001 has shipped and is currently in transit with ACME Express. You can expect it to arrive in about 2 days.

Let me know if there's anything else I can help you with!

Warmly,  
Aurora  
Customer Support


## QA / summary chain

In [3]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the support interaction in one line for the CRM: intent, action, outcome."),
    ("human", "Customer: {ticket}\n\nAgent reply: {reply}"),
])

summary_chain = summary_prompt | llm_noreason | StrOutputParser()
print(summary_chain.invoke({"ticket": "Where is my order 1001?", "reply": reply}))

Customer inquired about order status; agent confirmed shipment and provided tracking details; customer received expected delivery timeline.


### Definition of done
- `draft_chain` produces an on-brand reply grounded only in the research.
- `summary_chain` produces a one-line CRM summary.
- Bonus: a tone-check chain that flags replies that sound rude or off-brand.